# 02 · AutoML for scientific classification

**ML Science Lab · hands-on notebook**

**Learning goals:** compare a baseline with automated model/hyperparameter selection, control compute time, and evaluate once on a reserved test set.

You can work in pairs. Each pair chooses one budget of **10, 30 or 90 seconds**; compare results as a group, rather than running all budgets individually.

### 0. Install FLAML (Colab CPU)

Run once. If the Colab runtime has been restarted, rerun this installation. FLAML's AutoML extras install additional dependencies.

In [ ]:
%pip -q install "flaml[automl]>=2,<3"

### 1. Load chemistry data and reserve the test set

The untouched 20% test partition will not be used in cross-validation or model selection.

In [ ]:
import time
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from flaml import AutoML

data = load_wine(as_frame=True)
X, y = data.data, data.target
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print('Development:', len(X_dev), '| Untouched test:', len(X_test))

### 2. Baseline performance on development CV

We use the same CV folds when evaluating the fixed random forest model. This makes the baseline more informative than training accuracy.

In [ ]:
baseline = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=2)
fold_scores = cross_val_score(baseline, X_dev, y_dev, cv=cv, scoring='accuracy', n_jobs=1)
print('Fixed random-forest CV accuracy:', fold_scores.round(3))
print('Mean CV accuracy:', round(fold_scores.mean(), 3))

### 3. Automated model and hyperparameter selection

Set the time budget below. FLAML explores several estimator families and configurations on **development data only**. With very small data, selection is noisy; don't overinterpret tiny score differences.

In [ ]:
TIME_BUDGET = 90  # YOUR TURN: choose 10, 30 or 90 (seconds)

automl = AutoML()
start = time.perf_counter()
automl.fit(
    X_train=X_dev, y_train=y_dev,
    task='classification', metric='accuracy',
    time_budget=TIME_BUDGET,
    eval_method='cv', n_splits=3,
    estimator_list=['rf', 'extra_tree', 'lgbm'],
    seed=42, n_jobs=2, verbose=0,
)
print(f'Time used: {time.perf_counter() - start:.1f} seconds')
print('Selected estimator:', automl.best_estimator)
print('Best configuration:', automl.best_config)
print('Approx. internal CV accuracy:', round(1 - automl.best_loss, 3))

### 4. Final evaluation — once, after selection

Evaluate the pre-chosen baseline and the selected AutoML model once on **the untouched test set**. Afterwards, don't use these scores to select another configuration.

In [ ]:
baseline.fit(X_dev, y_dev)
for name, estimator in [('Fixed random forest', baseline), ('FLAML AutoML', automl)]:
    predictions = estimator.predict(X_test)
    print(f'{name:20s} test accuracy={accuracy_score(y_test, predictions):.3f}; '
          f'test macro-F1={f1_score(y_test, predictions, average="macro"):.3f}')

### Reflection

- Did longer search always improve **development CV** accuracy? (Compare across different groups.)
- Did the best CV configuration necessarily generalize better?
- How many trials did AutoML effectively conduct compared with a manually chosen baseline?
- What would need to change for scientific experiments with repeated measurements or temporal dependence?

**Caution:** the tiny Wine dataset produces high-variance results. This is a method demonstration, not a claim that a given model is best for chemistry tasks.